In [2]:
import pandas as pd
from pathlib import Path

In [3]:
#Read and rename the columns of the dataset

data_file_path = "/Users/lipikam/Desktop/MinorProject/ANSC4040MiniProject/project_1_ANSC_4040_dataset.csv"

df = pd.read_csv(data_file_path)

df = df.rename(columns={
    "Avgmilkflow": "AverageMilkFlowKgPerMin",
    "Flow30_60Session": "MilkFlow30To60SecondsKgPerMin",
    "YieldFirst2Minute_Session": "MilkYieldFirst2MinutesKg",
    "YieldSession": "TotalMilkYieldSessionKg",
    "DurationSession_sec": "MilkingDurationSessionSeconds",
    "milking": "MilkingSession"
})

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

Rows: 8,495,421
Columns: 10


/var/folders/wm/1jg9q6hn6qb190_w715mhp100000gn/T/ipykernel_8791/1709508777.py:5: DtypeWarning: Columns (0: ReproductionStatus) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_file_path)


,AnimalNumber,LactationNumber,DaysInMilk,ReproductionStatus,AverageMilkFlowKgPerMin,MilkFlow30To60SecondsKgPerMin,YieldFirst2Min_Session,TotalMilkYieldSessionKg,MilkingDurationSessionSeconds,MilkingSession
0,31317.0,2.0,255.0,Pregnant,2.812273,2.898455,5.874021,8.028585,168,2
1,60614.0,1.0,251.0,Pregnant,2.902991,3.801104,6.372973,7.030682,143,1
2,36947.0,1.0,183.0,Pregnant,3.401943,3.701314,7.048825,16.374685,285,3
3,31720.0,2.0,82.0,Bred,4.309128,4.799007,8.876803,18.461209,256,1
4,23630.0,6.0,182.0,Pregnant,3.719457,3.701314,7.874364,13.199538,210,3


In [4]:
#Create a metadata table for the dataset

metadata_file_path = "/Users/lipikam/Desktop/MinorProject/MiniProjectMetaData.xlsx"
output_file_path = "/Users/lipikam/Desktop/MinorProject/MiniProjectMetaData.xlsx"

# Read the metadata definitions
df_metadata = pd.read_excel(metadata_file_path)

# Units based on the variable definitions and column names
canonical_units = {
    "AnimalNumber": "Identifier",
    "LactationNumber": "Count",
    "DaysInMilk": "Days",
    "ReproductionStatus": "Category",
    "AverageMilkFlowKgPerMin": "kg/min",
    "MilkFlow30To60SecondsKgPerMin": "kg/min",
    "MilkYieldFirst2MinutesKg": "kg",
    "TotalMilkYieldSessionKg": "kg",
    "MilkingDurationSessionSeconds": "seconds",
    "MilkingSession": "Category"
}

metadata_rows = []

for _, metadata_row in df_metadata.iterrows():
    current_name = metadata_row["Current name"]
    suggested_name = metadata_row["Suggested name"]

    # Use the current name before renaming, or suggested name after renaming
    if suggested_name in df.columns:
        column_name = suggested_name
    elif current_name in df.columns:
        column_name = current_name
    else:
        continue

    series = df[column_name]
    numeric_series = pd.to_numeric(series, errors="coerce")

    if pd.api.types.is_numeric_dtype(series):
        minimum = numeric_series.min()
        maximum = numeric_series.max()
        value_range = maximum - minimum
    else:
        minimum = "N/A"
        maximum = "N/A"
        value_range = "N/A"

    example_values = ", ".join(
        series.dropna().astype(str).unique()[:5]
    )

    metadata_rows.append({
        "Current name": current_name,
        "Suggested name": suggested_name,
        "Definition": metadata_row["Definition"],
        "Total count": len(series),
        "Non-missing count": series.notna().sum(),
        "Missing count": series.isna().sum(),
        "Missing percent": round(series.isna().mean() * 100, 2),
        "Data type": str(series.dtype),
        "Unique values": series.nunique(dropna=True),
        "Minimum": minimum,
        "Maximum": maximum,
        "Range": value_range,
        "Canonical unit": canonical_units.get(suggested_name, "To confirm"),
        "Example values": example_values
    })

df_metadata_completed = pd.DataFrame(metadata_rows)

display(df_metadata_completed)

,Current name,Suggested name,Definition,Total count,Non-missing count,Missing count,Missing percent,Data type,Unique values,Minimum,Maximum,Range,Canonical unit,Example values
0,AnimalNumber,AnimalNumber,Animal identifier,8495421,6794418,1701003,20.02,float64,9087,11038.0,60630.0,49592.0,Identifier,"31317.0, 60614.0, 36947.0, 31720.0, 23630.0"
1,LactationNumber,LactationNumber,Lactation number,8495421,6794418,1701003,20.02,float64,12,1.0,12.0,11.0,Count,"2.0, 1.0, 6.0, 4.0, 3.0"
2,DaysInMilk,DaysInMilk,Days since calving,8495421,6794414,1701007,20.02,float64,870,1.0,870.0,869.0,Days,"255.0, 251.0, 183.0, 82.0, 182.0"
3,ReproductionStatus,ReproductionStatus,Reproductive-status category,8495421,6794418,1701003,20.02,str,4,N/A,N/A,N/A,Category,"Pregnant, Bred, Open, Fresh"
4,Avgmilkflow,AverageMilkFlowKgPerMin,Average milk flow rate,8495421,8495249,172,0.00,float64,219,0.0,23.496085,23.496085,kg/min,"2.812272694, 2.902991168, 3.401942775, 4.30912..."
5,Flow30_60Session,MilkFlow30To60SecondsKgPerMin,Flow rate from 30–60 seconds into milking,8495421,8495421,0,0.00,float64,102,0.0,10.500663,10.500663,kg/min,"2.8984552443, 3.8011040606000006, 3.7013137392..."
6,YieldSession,TotalMilkYieldSessionKg,Total milk produced during the session,8495421,8495421,0,0.00,float64,1010,5.034875,54.839318,49.804442,kg,"8.028584949, 7.030681735, 16.374684557000002, ..."
7,DurationSession_sec,MilkingDurationSessionSeconds,Session duration,8495421,8495421,0,0.00,int64,604,100,785,685,seconds,"168, 143, 285, 256, 210"
8,milking,MilkingSession,Milking/session indicator or category,8495421,8495421,0,0.00,int64,3,1,3,2,Category,"2, 1, 3"


In [5]:
df_metadata_completed.to_excel(
    output_file_path,
    sheet_name="Metadata",
    index=False
)

print(f"Saved completed metadata to:\n{output_file_path}")

Saved completed metadata to:
/Users/lipikam/Desktop/MinorProject/MiniProjectMetaData.xlsx
